In [ ]:
import os
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from netin.models import PATCHModel, CompoundLFM
from scipy.stats import gaussian_kde

from patch.constants import PATH_INFERENCE_VALIDATION, MAP_MODEL_COLOR, SIZE_FIG, MAP_LFM_SHORT, L_HOMOPHILY, L_TAU as L_TAU_ORIG, PATH_PLOTS, N_SAMPLES
from patch.statistics import get_cdf, compute_contour_lines

In [ ]:
# Set figure size
plt.rcParams["figure.figsize"] = SIZE_FIG
# Remove legend border
plt.rcParams["legend.frameon"] = False
# Remove top and right axis
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
PATH_PREFIX = "7r-smc_"
LFM_COMBS = [
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value),
    (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.PAH.value, CompoundLFM.PAH.value),
]
L_TAU = [L_TAU_ORIG[0], L_TAU_ORIG[3]]


In [ ]:
folder_plots = os.path.join("../", PATH_PLOTS, "validation", PATH_PREFIX)
if not os.path.exists(folder_plots):
    os.makedirs(folder_plots)
print(f"Saving plots to {folder_plots}")

In [ ]:
def create_true_config_folder_path(
        h_true: float, tau_true: float,
        lfm_global_true: str, lfm_tc_true: str):
    return os.path.join(
        "../",
        PATH_INFERENCE_VALIDATION,
        (f"{PATH_PREFIX}"
         f"lfm-g-true-{lfm_global_true}_lfm-t-true-{lfm_tc_true}_"
         f"h-true-{h_true}_tau-true-{tau_true}"
         "/"))

In [ ]:
def create_inf_folder_name(
        lfm_global_inf: str, lfm_tc_inf: str):
    return (f"lfm-g-inf-{lfm_global_inf}_lfm-t-inf-{lfm_tc_inf}/")

In [ ]:
def load_posterior(
    h_true:float, tau_true: float,
    lfm_global_true: str, lfm_tc_true: str,
    lfm_global_inf: str, lfm_tc_inf: str
) -> np.ndarray:
    posteriors = np.load(
        os.path.join(
            create_true_config_folder_path(
                h_true=h_true, tau_true=tau_true,
                lfm_global_true=lfm_global_true,
                lfm_tc_true=lfm_tc_true),
            create_inf_folder_name(
                lfm_global_inf=lfm_global_inf,
                lfm_tc_inf=lfm_tc_inf),
            f"posteriors.npz"))
    return posteriors

In [ ]:
def style_figure(
        fig: Any,
        lfm_global_true: str, lfm_tc_true: str):
    legend = fig.legend(
        handles=[
            plt.Line2D(
                [0], [0],
                color=MAP_MODEL_COLOR[lfm_global, lfm_tc],
                label=f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$")
            for lfm_global, lfm_tc in LFM_COMBS
        ],
        # Position legend outside in a horizontal line
        loc="upper center",
        ncol=len(LFM_COMBS),
        bbox_to_anchor=(0.5, 1.05),
        # Reduce length of lines in legend
        handlelength=0.5)

    fig.text(0.15, .9,
        f"${MAP_LFM_SHORT[lfm_global_true]},{MAP_LFM_SHORT[lfm_tc_true]}$")

    fig.tight_layout()

In [ ]:
summary = np.load(
    os.path.join(
        create_true_config_folder_path(
            h_true=0.01, tau_true=0.,
            lfm_global_true=CompoundLFM.HOMOPHILY.value,
            lfm_tc_true=CompoundLFM.UNIFORM.value),
        f"summary.npz"
    )
)
summary.files

In [ ]:
posteriors = np.load(
    os.path.join(
        create_true_config_folder_path(
            h_true=0.01, tau_true=0.,
            lfm_global_true=CompoundLFM.HOMOPHILY.value,
            lfm_tc_true=CompoundLFM.UNIFORM.value),
        create_inf_folder_name(
            lfm_global_inf=CompoundLFM.HOMOPHILY.value,
            lfm_tc_inf=CompoundLFM.UNIFORM.value),
        f"posteriors.npz"
    )
)
posteriors.files

## Model Selection

In [ ]:
posterior = load_posterior(
    h_true=0.01, tau_true=0.,
    lfm_global_true=CompoundLFM.HOMOPHILY.value,
    lfm_tc_true=CompoundLFM.UNIFORM.value,
    lfm_global_inf=CompoundLFM.HOMOPHILY.value,
    lfm_tc_inf=CompoundLFM.UNIFORM.value)
posteriors.files

In [ ]:
dist = posterior["discrepancies"]
dist.shape

In [ ]:
def draw_discrepancy_distribution(
        lfm_global_true: str, lfm_tc_true: str,
) -> Tuple[Any, Any]:
    fig, a_ax = plt.subplots(
        nrows=len(L_TAU),
        ncols=len(L_HOMOPHILY),
        sharex="row",
        sharey="row",
    )
    for i, tau_true in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
                l_posteriors = [load_posterior(
                    h_true=h_true, tau_true=tau_true,
                    lfm_global_true=lfm_global_true,
                    lfm_tc_true=lfm_tc_true,
                    lfm_global_inf=lfm_global_inf,
                    lfm_tc_inf=lfm_tc_inf)["discrepancies"]\
                        for lfm_global_inf, lfm_tc_inf in LFM_COMBS]

                bin_edges = np.linspace(np.min(l_posteriors), np.max(l_posteriors), 50)
                l_hist = [np.histogram(
                    a_post,
                    bins=bin_edges,
                    density=True)[0] for a_post in l_posteriors]

                for hist, post, (lfm_g, lfm_t) in zip(l_hist, l_posteriors, LFM_COMBS):
                    a_ax[i,j].bar(
                        bin_edges[:-1],
                        hist,
                        width=np.diff(bin_edges),
                        color=MAP_MODEL_COLOR[lfm_g, lfm_t],
                        alpha=0.5,
                    )
                    a_ax[i,j].axvline(
                        x=post.mean(),
                        color=MAP_MODEL_COLOR[lfm_g, lfm_t],
                        linestyle="--",
                    )


    for ax in a_ax[:, 0]:
        ax.set_ylabel("Density")
    for ax in a_ax[-1, :]:
        ax.set_xlabel("$d_{{euc}}$")
    style_figure(fig, lfm_global_true, lfm_tc_true)
    fig.tight_layout()
    return fig, a_ax

In [ ]:
for lfm_global_true, lfm_tc_true in LFM_COMBS:
    print(f"Plotting {lfm_global_true}, {lfm_tc_true}")
    f, a = draw_discrepancy_distribution(lfm_global_true, lfm_tc_true)
    path_file = os.path.join(
        folder_plots,
        f"discrepancy_lfm-g-{lfm_global_true}_lfm-tc-{lfm_tc_true}.pdf")
    f.savefig(os.path.join(path_file))
    print(f"Saved to {path_file}")

### Confusion Matrix

In [ ]:
def compute_inferred_mwu(
        lfm_global_true: str, lfm_tc_true: str,
        h_true: float, tau_true: float) -> np.ndarray:
    a_dist_inf = np.zeros((len(LFM_COMBS), N_SAMPLES))
    a_mw = np.zeros(len(LFM_COMBS))
    for j, (lfm_g_inf, lfm_tc_inf) in enumerate(LFM_COMBS):
        a_dist_inf[j] = load_posterior(
            h_true=h_true, tau_true=tau_true,
            lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true,
            lfm_global_inf=lfm_g_inf, lfm_tc_inf=lfm_tc_inf)["discrepancies"]
    for j, _ in enumerate(LFM_COMBS):
        a_inf = a_dist_inf[j]
        a_inf_others = np.concatenate(
            [a_dist_inf[k] for k in range(len(LFM_COMBS)) if k != j])
        a_mw[j] = sc.stats.mannwhitneyu(
            a_inf_others, a_inf).statistic / (N_SAMPLES * N_SAMPLES * 3)
    return a_mw

In [ ]:
# Plot a confusion matrix for a single parameter combination `h`, `tau`
# The confusion matrix is a 4 x 4 matrix where the rows are the true model
# and the columns are the inferred model
# The elements are test statistic of the Mann-Whitney U test between the
# distances of the given inferred model to all other inferred model

def draw_confusion_matrix(
        h_true: float, tau_true: float,
        tpl_fig_ax: Optional[Tuple[Any, Any]] = None,
        is_subplot: bool = False) -> Tuple[Any, Any]:
    fig, ax = plt.subplots() if tpl_fig_ax is None else tpl_fig_ax
    a_matrix = np.zeros((len(LFM_COMBS), len(LFM_COMBS)))
    for i, (lfm_g_true, lfm_tc_true) in enumerate(LFM_COMBS):
        a_matrix[i] = compute_inferred_mwu(
            lfm_global_true=lfm_g_true, lfm_tc_true=lfm_tc_true,
            h_true=h_true, tau_true=tau_true)

    cbar = ax.imshow(a_matrix, cmap="Blues", vmin=0, vmax=1, origin="lower")

    # For each row, mark the maximum value
    # with a border inside the confusion matrix cell
    off_width = .0165
    for i, _ in enumerate(LFM_COMBS):
        j_max = np.argmax(a_matrix[i])
        ax.add_patch(plt.Rectangle(
            (j_max - 0.5 + off_width, i - 0.5 + off_width),
            width=1 - (2 * off_width),
            height=1 - (2 * off_width),
            fill=False,
            edgecolor=plt.colormaps["tab20"](2),
            linewidth=2))

    # Slightly extend the axes to make the borders visible
    # ax.set_xlim(-0.5, len(LFM_COMBS) - 1 + 0.5)
    # ax.set_ylim(-0.5, len(LFM_COMBS) - 1 + 0.5)

    if not is_subplot:
        fig.colorbar(cbar)
        ax.set_xticks(range(len(LFM_COMBS)))
        ax.set_yticks(range(len(LFM_COMBS)))
        ax.set_xticklabels([f"${MAP_LFM_SHORT[lfm_g]},{MAP_LFM_SHORT[lfm_t]}$"\
            for lfm_g, lfm_t in LFM_COMBS])
        ax.set_yticklabels([f"${MAP_LFM_SHORT[lfm_g]},{MAP_LFM_SHORT[lfm_t]}$"\
            for lfm_g, lfm_t in LFM_COMBS])
        ax.set_xlabel("Inferred model")
        ax.set_ylabel("True model")


    return fig, ax



In [ ]:
draw_confusion_matrix(
    h_true=0.01, tau_true=0.75)


In [ ]:
# Iterate over all parameter combinations and plot the confusion matrix
# Plot a single colorbar for all confusion matrices at the right

fig, a_ax = plt.subplots(
    nrows=len(L_TAU),
    ncols=len(L_HOMOPHILY),
    sharex=True,
    sharey=True,
)
for i, tau_true in enumerate(L_TAU):
    for j, h_true in enumerate(L_HOMOPHILY):
        draw_confusion_matrix(
            h_true=h_true, tau_true=tau_true,
            tpl_fig_ax=(fig, a_ax[i, j]),
            is_subplot=True)
for ax in a_ax[-1, :]:
    ax.set_xticks(range(len(LFM_COMBS)))
    ax.set_xticklabels([f"${MAP_LFM_SHORT[lfm_g]},{MAP_LFM_SHORT[lfm_t]}$"\
        for lfm_g, lfm_t in LFM_COMBS], rotation=90)
    ax.set_xlabel("Inference")
for ax in a_ax[:, 0]:
    ax.set_yticks(range(len(LFM_COMBS)))
    ax.set_yticklabels([f"${MAP_LFM_SHORT[lfm_g]},{MAP_LFM_SHORT[lfm_t]}$"\
        for lfm_g, lfm_t in LFM_COMBS])
    ax.set_ylabel("True model")
for tau_true, y_pos in zip(L_TAU, (.8225, .5625)):
    fig.text(
        .05, y_pos,
        f"$\\tau={tau_true}$",
        transform=fig.transFigure)
for h_true, x_pos in zip(L_HOMOPHILY, np.linspace(.185, .75, len(L_HOMOPHILY))):
    fig.text(
        x_pos, .825,
        f"$h={h_true}$",
        transform=fig.transFigure)

fig.text(0.875, 0.825, "MW-U")

fig.tight_layout()
fig.subplots_adjust(right=0.85, hspace=-.5)
cbar_ax = fig.add_axes([0.8625, 0.3625, 0.01, 0.435])
fig.colorbar(a_ax[0, 0].get_children()[0], cax=cbar_ax)
fig.savefig(os.path.join(folder_plots, "confusion_matrix.pdf"))
print(f"Saved to {os.path.join(folder_plots, 'confusion_matrix.pdf')}")

## Parameter Inference

### Posterior plot

In [ ]:
def compute_2d_hists(
        lfm_global_true: str, lfm_tc_true: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    a_posteriors = np.zeros(
        (len(L_TAU), len(L_HOMOPHILY), 2, N_SAMPLES))
    a_model_selection = np.zeros(
        (len(L_TAU), len(L_HOMOPHILY)), dtype=int)
    for i, tau_true in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
            a_mwu = compute_inferred_mwu(
                lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true,
                h_true=h_true, tau_true=tau_true)
            a_model_selection[i, j] = np.argmax(a_mwu)
            lfm_g_inf, lfm_t_inf = LFM_COMBS[a_model_selection[i, j]]
            posterior = load_posterior(
                h_true=h_true, tau_true=tau_true,
                lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true,
                lfm_global_inf=lfm_g_inf, lfm_tc_inf=lfm_t_inf)
            a_posteriors[i, j, 0] = posterior["tau"]
            a_posteriors[i, j, 1] = posterior["h"]

            # print(f"DEBUGGING Setting tau = 1")
            # a_posteriors[i, j, 0] = 1

    a_hist = np.zeros(
        (len(L_TAU), len(L_HOMOPHILY), 25, 25))
    for i, tau_true in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
            hist, x_edges, y_edges = np.histogram2d(
                a_posteriors[i, j, 0],
                a_posteriors[i, j, 1],
                bins=25,
                range=[[0, 1], [0, 1]],
                density=True)
            a_hist[i, j] = hist
    return a_hist, x_edges, y_edges, a_model_selection

In [ ]:
# For each true model, plot the estimated posterior of the inferred model with the lowest distance

def draw_single_posterior(
        a_joint_posterior: np.ndarray,
        vmin: float, vmax: float,
        h_true: float, tau_true: float,
        tpl_fig_ax: Optional[Tuple[Any, Any]] = None,
        is_subplot: bool = False) -> Tuple[Any, Any]:
    fig, ax = plt.subplots() if tpl_fig_ax is None else tpl_fig_ax

    ax.imshow(
        a_joint_posterior.T,
        cmap="Blues",
        origin="lower",
        extent=[0, 1, 0, 1],
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
    )

    ax.axvline(x=tau_true, color=plt.colormaps["tab20"](2), linestyle="--")
    ax.axhline(y=h_true, color=plt.colormaps["tab20"](2), linestyle="--")
    if not is_subplot:
        ax.set_xlabel("$\\tau$")
        ax.set_ylabel("$h$")
    ax.set_xlim(-.025, 0.75 + .025)
    ax.set_ylim(-.0125, 1 + .0125)

    fig.tight_layout()
    return fig, a_ax

In [ ]:
def draw_single_contour_plot(
        a_tau: np.ndarray, a_h: np.ndarray,
        ax: plt.Axes,
        percentiles: List[float],
        plot_clabel: bool = True):
    X, Y, Z, thresholds = compute_contour_lines(
        a_tau=a_tau,
        a_h=a_h,
        percentiles=percentiles,
    )

    # Create the contour plot
    for threshold, percentile in zip(
            thresholds, percentiles):
        contour = ax.contour(
            X, Y, Z,
            levels=[threshold],
            colors=[plt.colormaps["Blues"](.75)],
            # colors=[plt.colormaps["tab20"](2)],
            # colors=[plt.colormaps["Greys"](percentile)],
            # alpha=.75,  # Adjust alpha based on percentile
        )
        if plot_clabel:
            ax.clabel(
                contour,
                inline=True,
                rightside_up=True,
                inline_spacing=2.5,
                fontsize=8,
                fmt=lambda _: f"{percentile:.0%}")

In [ ]:
def draw_lfm_model_posterior(
        lfm_global_true: str, lfm_tc_true: str) -> Tuple[Any, Any]:
    fig, a_ax = plt.subplots(
            nrows=len(L_TAU),
            ncols=len(L_HOMOPHILY),
            sharex=True,
            sharey=True)


    a_hist, _, _, a_model_sel = compute_2d_hists(
        lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true)
    vmin = np.min(a_hist)
    vmax = np.max(a_hist)

    for i, tau_true in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
            draw_single_posterior(
                a_joint_posterior=a_hist[i, j],
                vmin=vmin, vmax=vmax,
                h_true=h_true, tau_true=tau_true,
                tpl_fig_ax=(fig, a_ax[i, j]),
                is_subplot=True)

            lfm_global_inf, lfm_tc_inf = LFM_COMBS[a_model_sel[i, j]]
            posteriors = load_posterior(
                h_true=h_true, tau_true=tau_true,
                lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true,
                lfm_global_inf=lfm_global_inf, lfm_tc_inf=lfm_tc_inf,)
            h = posteriors["h"]
            tau = posteriors["tau"]

            # Draw the contour lines
            draw_single_contour_plot(
                a_tau=tau,
                a_h=h,
                ax=a_ax[i, j],
                percentiles=[.95],
                plot_clabel=(i == 1 and j == 2))

    for i, _ in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
            lfm_g_inf, lfm_t_inf = LFM_COMBS[a_model_sel[i, j]]
            right = (i == 0)
            top = (j < (len(L_HOMOPHILY) - 1))
            # y_off = .1 if i == 0 else .02
            a_ax[i, j].text(
                # .99 if i == 0 else .01,
                .99 if right else .01,
                .99 if top else .01,
                # .25 if i == 0 else .05,
                # .9,
                f"${MAP_LFM_SHORT[lfm_g_inf]},{MAP_LFM_SHORT[lfm_t_inf]}$",
                verticalalignment="top" if top else "bottom",
                horizontalalignment="right" if right else "left",
                fontsize=10,)

    for ax in a_ax[:, 0]:
        ax.set_ylabel("$h_{{inf}}$")
        ax.set_yticks(L_HOMOPHILY)
        # ax.set_yticks(y_edges)
    for ax in a_ax[-1, :]:
        ax.set_xlabel("$\\tau_{{inf}}$")
        # ax.set_xticks(x_edges)
        ax.set_xticks(L_TAU_ORIG)
        ax.set_xticklabels([tau if tau in L_TAU else None for tau in L_TAU_ORIG])

    # Write the true values
    for ax, tau_true in zip(a_ax[:,0], L_TAU):
        ax.text(
            tau_true + 0.025, .95,
            f"$\\tau_{{true}}={tau_true}$",
            horizontalalignment="left",
            verticalalignment="top",
            rotation=90,
            color=plt.colormaps["tab20"](2),
            fontsize=8)
    for j, (h_true, ax) in enumerate(
            zip(L_HOMOPHILY,
                a_ax[0, :])):
                # np.linspace(.125, .75, len(L_HOMOPHILY)))):
        ax.text(
            .99, h_true + (.02 if j < len(L_HOMOPHILY) - 1 else -.02),
            f"$h_{{true}}={h_true:.2f}$",
            fontsize=8,
            horizontalalignment="right",
            verticalalignment="bottom" if j < len(L_HOMOPHILY) - 1 else "top",
            color=plt.colormaps["tab20"](2),)

    # Plot colorbar to the right
    fig.tight_layout()
    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.8625, 0.3625, 0.01, 0.435])
    fig.colorbar(a_ax[0, 0].get_children()[0], cax=cbar_ax)
    return fig, a_ax

In [ ]:
draw_lfm_model_posterior(
    lfm_global_true=CompoundLFM.HOMOPHILY.value,
    lfm_tc_true=CompoundLFM.UNIFORM.value)


In [ ]:
for lfm_global_true, lfm_tc_true in LFM_COMBS:
    print(f"Plotting {lfm_global_true}, {lfm_tc_true}")
    f, a = draw_lfm_model_posterior(lfm_global_true, lfm_tc_true)
    # f.tight_layout()
    path_file = os.path.join(
        folder_plots,
        f"posterior_lfm-g-{lfm_global_true}_lfm-t-{lfm_tc_true}.pdf")

    f.savefig(os.path.join(path_file))
    print(f"Saved to {path_file}")

### Contour lines

In [ ]:
def draw_contour_lines(h: np.ndarray, tau: np.ndarray,
                       lfm_global: str, lfm_tc: str,
                       tpl_fig_ax: Optional[Tuple[Any, Any]] = None,
                       is_subplot: bool = False)\
        -> Tuple[Any, Any]:
    fig, ax = plt.subplots() if tpl_fig_ax is None else tpl_fig_ax

    # Stack the data for the KDE: shape should be (n_dim, n_samples)
    data = np.vstack([h, tau])
    kde = gaussian_kde(data)

    # Create a grid over the range of the data
    X, Y = np.mgrid[0:1:250j, 0:1:250j]
    positions = np.vstack([X.ravel(), Y.ravel()])

    # Evaluate the KDE on the grid and reshape the result back into a 2D array
    Z = np.reshape(kde(positions), X.shape)

    # Create the contour plot
    contour = ax.contour(
        X, Y, Z,
        levels=[3, 6, 9],
        colors=[
            MAP_MODEL_COLOR[lfm_global, lfm_tc]])
    ax.clabel(contour, inline=True, fontsize=8)

    if not is_subplot:
        ax.set_xlabel('$h$')
        ax.set_ylabel('$\\tau$')
    return fig, ax


In [ ]:
def draw_single_model_contours(
        lfm_global_true: str, lfm_tc_true: str,
        h_true: float, tau_true: float,
        tpl_fig_ax: Optional[Tuple[Any, Any]] = None,
        is_subplot: bool = False)\
        -> Tuple[Any, Any]:
    for lfm_global_inf, lfm_tc_inf in LFM_COMBS:
        posteriors = load_posterior(
            h_true=h_true, tau_true=tau_true,
            lfm_global_true=lfm_global_true, lfm_tc_true=lfm_tc_true,
            lfm_global_inf=lfm_global_inf, lfm_tc_inf=lfm_tc_inf)

        h = posteriors["h"]
        tau = posteriors["tau"]

        tpl_fig_ax = draw_contour_lines(
            h, tau,
            lfm_global=lfm_global_inf,
            lfm_tc=lfm_tc_inf,
            tpl_fig_ax=tpl_fig_ax,
            is_subplot=is_subplot)

    _, ax = tpl_fig_ax
    ax.axhline(y=tau_true, color="black", linestyle="--")
    ax.axvline(x=h_true, color="black", linestyle="--")
    ax.set_xlim(-.05, 1)
    ax.set_ylim(-.05, 1)
    return tpl_fig_ax

In [ ]:
fig, _ = draw_single_model_contours(
    lfm_global_true=CompoundLFM.HOMOPHILY.value,
    lfm_tc_true=CompoundLFM.UNIFORM.value,
    h_true=0.01,
    tau_true=0.)
style_figure(fig, CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value)


In [ ]:

def draw_lfm_model_contour(
        lfm_global_true: str, lfm_tc_true: str) -> Tuple[Any, Any]:
    fig, a_ax = plt.subplots(
        nrows=len(L_TAU),
        ncols=len(L_HOMOPHILY),
        sharex=True,
        sharey=True,
    )

    for i, tau_true in enumerate(L_TAU):
        for j, h_true in enumerate(L_HOMOPHILY):
            draw_single_model_contours(
                lfm_global_true=lfm_global_true,
                lfm_tc_true=lfm_tc_true,
                h_true=h_true,
                tau_true=tau_true,
                tpl_fig_ax=(fig, a_ax[i, j]),
                is_subplot=True)

    style_figure(fig, lfm_global_true, lfm_tc_true)

    # Add labels
    for i, tau_true in enumerate(L_TAU):
        a_ax[i, 0].set_ylabel(f"$\\tau$")
    for j, h_true in enumerate(L_HOMOPHILY):
        a_ax[-1, j].set_xlabel(f"$h$")

    return fig, a_ax

In [ ]:
for lfm_global_true, lfm_tc_true in LFM_COMBS:
    print(f"Plotting {lfm_global_true}, {lfm_tc_true}")
    f, a = draw_lfm_model_contour(lfm_global_true, lfm_tc_true)
    f.tight_layout()
    path_file = os.path.join(
        folder_plots,
        f"contour_lfm-g-{lfm_global_true}_lfm-tc-{lfm_tc_true}.pdf")
    f.savefig(os.path.join(path_file))
    print(f"Saved to {path_file}")